# Clinical Documentation Assistant


In [12]:
from google.colab import drive
import os
import glob

drive.mount('/content/drive', force_remount=True)

# Reset về thư mục root của Colab
%cd /content/
if not os.path.exists('Clinical-Ambient-Documentation-Assistant'):
    !git clone https://github.com/DuyVuux/Clinical-Ambient-Documentation-Assistant.git

%cd /content/Clinical-Ambient-Documentation-Assistant

# Tìm file zip
possible_path = "/content/drive/MyDrive/Clinical Ambient Docs Assistant Data/data_lake.zip"
if not os.path.exists(possible_path):
    search = glob.glob("/content/drive/MyDrive/**/data_lake.zip", recursive=True)
    if search:
        possible_path = search[0]

if os.path.exists(possible_path):
    print(f"Đang giải nén dữ liệu từ: {possible_path}...")
    # Sử dụng tùy chọn -o để ghi đè và đảm bảo giải nén toàn bộ cấu trúc thư mục
    !unzip -o -q "{possible_path}" -d .
    print("Giải nén dữ liệu thành công!")
    # Kiểm tra lại xem thư mục audio đã xuất hiện chưa
    if os.path.exists('data/data_lake/silver/audio_clean'):
        print("Đã tìm thấy thư mục audio_clean.")
    else:
        print("CẢNH BÁO: Vẫn không thấy thư mục audio_clean sau khi giải nén!")
else:
    print("LỖI: Không tìm thấy file data_lake.zip!")

!pip install -r requirements.txt

Mounted at /content/drive
/content
/content/Clinical-Ambient-Documentation-Assistant
Đang giải nén dữ liệu từ: /content/drive/MyDrive/Clinical Ambient Docs Assistant Data/data_lake.zip...
Giải nén dữ liệu thành công!
Đã tìm thấy thư mục audio_clean.


## Thiết lập môi trường


In [13]:
import json
import os

# Đảm bảo đang ở đúng thư mục dự án
%cd /content/Clinical-Ambient-Documentation-Assistant

original_manifest = 'data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl'
fixed_manifest = 'data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1_colab.jsonl'

# Định nghĩa các prefix cũ và mới
old_prefix = '/home/duykhongngu28/massive/Clinical Ambient Documentation Assistant/'
new_prefix = '/content/Clinical-Ambient-Documentation-Assistant/'

if os.path.exists(original_manifest):
    os.makedirs(os.path.dirname(fixed_manifest), exist_ok=True)
    with open(original_manifest, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    with open(fixed_manifest, 'w', encoding='utf-8') as f:
        for line in lines:
            data = json.loads(line)
            # Cập nhật tất cả các trường có thể chứa đường dẫn
            for key in ['audio_filepath', 'raw_audio_path', 'raw_transcript_path']:
                if key in data and isinstance(data[key], str):
                    data[key] = data[key].replace(old_prefix, new_prefix)
            f.write(json.dumps(data, ensure_ascii=False) + '\n')
    print(f'Đã cập nhật manifest tại: {os.path.abspath(fixed_manifest)}')
else:
    print(f'LỖI: Không tìm thấy manifest gốc tại {original_manifest}.')

/content/Clinical-Ambient-Documentation-Assistant
Đã cập nhật manifest tại: /content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1_colab.jsonl


## Chuẩn bị dữ liệu


### Update Code from Git

To ensure you have the latest code without re-running the model predictions, execute the cell below. It will navigate to your project directory and pull any new changes from the `main` branch.

In [14]:
import os

repo_path = '/content/Clinical-Ambient-Documentation-Assistant'

if not os.path.exists(repo_path):
    print(f"Repository not found. Cloning...")
    %cd /content/
    !git clone https://github.com/DuyVuux/Clinical-Ambient-Documentation-Assistant.git
    %cd {repo_path}
else:
    %cd {repo_path}
    print("Cập nhật bản mới nhất từ Git (Ghi đè bản cục bộ)... ")
    !git fetch origin main
    !git reset --hard origin/main

print("Đã đồng bộ thành công với Git.")

/content/Clinical-Ambient-Documentation-Assistant
Cập nhật bản mới nhất từ Git (Ghi đè bản cục bộ)... 
From https://github.com/DuyVuux/Clinical-Ambient-Documentation-Assistant
 * branch            main       -> FETCH_HEAD
HEAD is now at eda53e0 feat(asr): add Zipformer RNN-T adapter and apply dynamic path resolution for Colab portability
Đã đồng bộ thành công với Git.


In [16]:
!pip install --upgrade transformers accelerate librosa datasets soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 31.7 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.9.0
    Uninstalling transformers-5.9.0:
      Successfully uninstalled transformers-5.9.0


In [17]:
import os
# Khắc phục lỗi 403 Forbidden và cảnh báo của Hugging Face
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = 'true'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = 'true'

# # Đường dẫn tuyệt đối đến file manifest đã sửa
# manifest_path = '/content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1_colab.jsonl'

# !git pull origin main

!python scripts/asr_eval/run_whisper_transformers.py \
    --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
    --model vinai/PhoWhisper-base \
    --output experiments/asr/baseline/predictions/dev/phowhisper_base_dev_predictions_20.jsonl \
    --max_samples 20


config.json: 100% 1.32k/1.32k [00:00<00:00, 491kB/s]
pytorch_model.bin: 100% 290M/290M [00:02<00:00, 104MB/s]
Loading weights: 100% 246/246 [00:00<00:00, 27803.04it/s]
generation_config.json: 100% 3.77k/3.77k [00:00<00:00, 11.4MB/s]
tokenizer_config.json: 100% 804/804 [00:00<00:00, 3.97MB/s]
vocab.json: 100% 836k/836k [00:00<00:00, 37.5MB/s]
tokenizer.json: 100% 2.20M/2.20M [00:00<00:00, 165MB/s]
merges.txt: 100% 494k/494k [00:00<00:00, 125MB/s]
normalizer.json: 100% 52.7k/52.7k [00:00<00:00, 67.7MB/s]
added_tokens.json: 100% 2.08k/2.08k [00:00<00:00, 8.17MB/s]
special_tokens_map.json: 100% 2.08k/2.08k [00:00<00:00, 8.16MB/s]
preprocessor_config.json: 100% 339/339 [00:00<00:00, 1.98MB/s]
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressToken

In [ ]:
!python scripts/asr_eval/run_whisper_transformers.py \
    --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
    --model vinai/PhoWhisper-base \
    --output experiments/asr/baseline/predictions/dev/phowhisper_base_dev_predictions.jsonl


Loading weights: 100% 246/246 [00:00<00:00, 22633.62it/s]
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBe

In [ ]:
!python scripts/asr_eval/run_whisper_transformers.py \
    --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
    --model openai/whisper-base \
    --output experiments/asr/baseline/predictions/dev/whisper_small_dev_predictions.jsonl

Loading weights: 100% 245/245 [00:00<00:00, 6782.55it/s]
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeg

In [ ]:
!python scripts/asr_eval/run_whisper_transformers.py \
    --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
    --model vinai/PhoWhisper-medium \
    --output experiments/asr/baseline/predictions/dev/phowhisper_medium_dev_predictions.jsonl


Loading weights: 100% 948/948 [00:00<00:00, 30743.42it/s]
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBe

In [ ]:
!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/baseline/predictions/dev/phowhisper_base_dev_predictions.jsonl \
  --output experiments/asr/baseline/metrics/dev/phowhisper_base_dev_metrics.json

{
  "n_samples": 200,
  "strict_wer": 0.2675344861084127,
  "normalized_wer": 0.24111132698659413,
  "strict_cer": 0.21052872930710323,
  "normalized_cer": 0.20731875085981566
}


In [ ]:
!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/baseline/predictions/dev/whisper_small_dev_predictions.jsonl \
  --output experiments/asr/baseline/metrics/dev/whisper_small_dev_metrics.json

{
  "n_samples": 200,
  "strict_wer": 0.5857781231785506,
  "normalized_wer": 0.5323489411307558,
  "strict_cer": 0.46691429357545744,
  "normalized_cer": 0.4484798459210345
}


In [ ]:
!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/baseline/predictions/dev/phowhisper_medium_dev_predictions.jsonl \
  --output experiments/asr/baseline/metrics/dev/phowhisper_medium_dev_metrics.json

{
  "n_samples": 200,
  "strict_wer": 0.246357101224014,
  "normalized_wer": 0.2150767437342141,
  "strict_cer": 0.19828495437244922,
  "normalized_cer": 0.19535011693492915
}


In [ ]:
!git pull origin main

From https://github.com/DuyVuux/Clinical-Ambient-Documentation-Assistant
 * branch            main       -> FETCH_HEAD
Updating cd657be..b459001
error: The following untracked working tree files would be overwritten by merge:
	experiments/asr/baseline/metrics/dev/chunkformer_ctc_large_vie_dev_10_metrics.json
	experiments/asr/baseline/metrics/dev/chunkformer_ctc_large_vie_dev_metrics.json
	experiments/asr/baseline/predictions/dev/chunkformer_ctc_large_vie_dev_10_predictions.jsonl
	experiments/asr/baseline/predictions/dev/chunkformer_ctc_large_vie_dev_predictions.jsonl
Please move or remove them before you merge.
Aborting


## 🧪 Bước 3: Chạy mô hình Baseline (XLSR & PhoWhisper)
Đây là các mô hình tiêu chuẩn để chúng ta so sánh hiệu suất với mô hình ChunkFormer.

In [ ]:
!pip install chunkformer

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 43.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 44.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.5/150.5 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 19.4 MB/s eta 0:00:00
  Created wheel for deepspeed: filename=deepspeed-0.19.1-py3-none-any.whl size=1851783 sha256=89b30e5a10ad1dcd96703cb82565909e421e7b7f5ae0301a854f3cb806bae9e6
  Stored in directory: /root/.cache/pip/wheels/ac/26/a4/6e4a074ecb7413ef47607c20bef4bcb2180a10330e5e844f92
  Created wheel for langid: filename=langid-1.1.6-py3-none

It seems the audio files are missing from the `audio_clean` directory. Let's inspect the contents of the `data/data_lake/silver/` directory to see if the `audio_clean` folder and its contents were extracted correctly.

In [ ]:
# List the contents of the silver directory
!ls -R /content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/

/content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/:
asr_manifests

/content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/asr_manifests:
ASR_MANIFEST_SCHEMA.md
asr_silver_manifest_v0_1.jsonl
common_voice_vi_dev_manifest_v0_1.jsonl
common_voice_vi_test_manifest_v0_1.jsonl
common_voice_vi_train_manifest_v0_1.jsonl
splits

/content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/asr_manifests/splits:
vietmed_dev_v0_1.jsonl	vietmed_train_candidate_v0_1.jsonl


In [ ]:
!python scripts/asr_eval/run_chunkformer_ctc.py \
  --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
  --output experiments/asr/baseline/predictions/dev/chunkformer_ctc_large_vie_dev_10_predictions.jsonl \
  --max_samples 10

[INFO] Loading ChunkFormer model: khanhld/chunkformer-ctc-large-vie
Fetching 10 files: 100% 10/10 [00:00<00:00, 105649.97it/s]
Download complete: : 0.00B [00:00, ?B/s]
  0% 0/1 [00:00<?, ?it/s]
[1/10] public_vietmed_0884
  0% 0/1 [00:00<?, ?it/s]
[2/10] public_vietmed_0174
  0% 0/1 [00:00<?, ?it/s]
[3/10] public_vietmed_0148
  0% 0/1 [00:00<?, ?it/s]
[4/10] public_vietmed_0420
  0% 0/1 [00:00<?, ?it/s]
[5/10] public_vietmed_0962
  0% 0/1 [00:00<?, ?it/s]
[6/10] public_vietmed_0149
  0% 0/1 [00:00<?, ?it/s]
[7/10] public_vietmed_0976
  0% 0/1 [00:00<?, ?it/s]
[8/10] public_vietmed_0080
  0% 0/1 [00:00<?, ?it/s]
[9/10] public_vietmed_0776
  0% 0/1 [00:00<?, ?it/s]
[10/10] public_vietmed_0303
[DONE] wrote predictions to experiments/asr/baseline/predictions/dev/chunkformer_ctc_large_vie_dev_10_predictions.jsonl


In [ ]:
# Evaluate WER for both 10-sample smoke test and the full 200-sample run
print("--- Metrics for 10 samples ---")
!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/baseline/predictions/dev/chunkformer_ctc_large_vie_dev_10_predictions.jsonl \
  --output experiments/asr/baseline/metrics/dev/chunkformer_ctc_large_vie_dev_10_metrics.json

print("\n--- Metrics for full 200 samples ---")
!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/baseline/predictions/dev/chunkformer_ctc_large_vie_dev_predictions.jsonl \
  --output experiments/asr/baseline/metrics/dev/chunkformer_ctc_large_vie_dev_metrics.json

--- Metrics for 10 samples ---
{
  "n_samples": 10,
  "strict_wer": 0.14901960784313725,
  "normalized_wer": 0.14901960784313725,
  "strict_cer": 0.14681440443213298,
  "normalized_cer": 0.14681440443213298
}

--- Metrics for full 200 samples ---
{
  "n_samples": 200,
  "strict_wer": 0.12900718865358463,
  "normalized_wer": 0.12900718865358463,
  "strict_cer": 0.12032833493832255,
  "normalized_cer": 0.12032833493832255
}


In [ ]:
import json
from pathlib import Path

# Chọn file bạn muốn kiểm tra (10 mẫu hoặc 200 mẫu)
# file_to_check = "experiments/asr/baseline/predictions/dev/chunkformer_ctc_large_vie_dev_10_predictions.jsonl"
file_to_check = "experiments/asr/baseline/predictions/dev/chunkformer_ctc_large_vie_dev_predictions.jsonl"

path = Path(file_to_check)

rows = []
if path.exists():
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    empty = [r for r in rows if not r.get("prediction_text", "").strip()]

    print(f"--- Đang kiểm tra file: {file_to_check} ---")
    print(f"Tổng số mẫu: {len(rows)}")
    print(f"Số dự đoán trống: {len(empty)}")

    print("\n--- Xem trước 3 mẫu đầu tiên ---")
    for r in rows[:3]:
        print("=" * 80)
        print("Sample ID:", r.get("sample_id"))
        print("REF:", r.get("reference_text"))
        print("PRED:", r.get("prediction_text"))
else:
    print(f"LỖI: Không tìm thấy file tại {path}")

--- Đang kiểm tra file: experiments/asr/baseline/predictions/dev/chunkformer_ctc_large_vie_dev_predictions.jsonl ---
Tổng số mẫu: 200
Số dự đoán trống: 0

--- Xem trước 3 mẫu đầu tiên ---
Sample ID: public_vietmed_0884
REF: nó sẽ giúp cải thiện được rất là nhiều các triệu chứng của người bệnh parkinson đặc biệt là các cái triệu chứng về vận
PRED: thích não sâu nó sẽ giúp cải thiện được rất là nhiều các triệu chứng của người bệnh parkinson đặc biệt là các cái triệu chứng
Sample ID: public_vietmed_0174
REF: đấy thì cái câu chuyện là gì ạ đạp xe ở trong nhà nó rất là an toàn nhưng mà đạp xe ra ngoài thì nó lại rất nhiều chuyện
PRED: ngoài đấy thì cái câu chuyện là gì ạ đạp xe ở trong nhà nó rất là an toàn nhưng mà đạp xe ra ngoài thì nó lại rất nhiều cái
Sample ID: public_vietmed_0148
REF: với tất cả những cái biểu hiện như vậy thì chúng tôi thấy khá là rõ ràng là nó có khả năng là cái biểu hiện của một cái
PRED: với tất cả những cái biểu hiện như vậy thì chúng tôi thấy khá là rõ ràng là 

## 📈 Tổng kết kết quả
So sánh các chỉ số giữa các mô hình để đưa ra kết luận về độ chính xác.

In [ ]:
!python scripts/asr_eval/run_chunkformer_ctc.py \
  --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
  --output experiments/asr/baseline/predictions/dev/chunkformer_ctc_large_vie_dev_predictions.jsonl

[INFO] Loading ChunkFormer model: khanhld/chunkformer-ctc-large-vie
Fetching 10 files: 100% 10/10 [00:00<00:00, 47446.88it/s]
Download complete: : 0.00B [00:00, ?B/s]
  0% 0/1 [00:00<?, ?it/s]
[1/200] public_vietmed_0884
  0% 0/1 [00:00<?, ?it/s]
[2/200] public_vietmed_0174
  0% 0/1 [00:00<?, ?it/s]
[3/200] public_vietmed_0148
  0% 0/1 [00:00<?, ?it/s]
[4/200] public_vietmed_0420
  0% 0/1 [00:00<?, ?it/s]
[5/200] public_vietmed_0962
  0% 0/1 [00:00<?, ?it/s]
[6/200] public_vietmed_0149
  0% 0/1 [00:00<?, ?it/s]
[7/200] public_vietmed_0976
  0% 0/1 [00:00<?, ?it/s]
[8/200] public_vietmed_0080
  0% 0/1 [00:00<?, ?it/s]
[9/200] public_vietmed_0776
  0% 0/1 [00:00<?, ?it/s]
[10/200] public_vietmed_0303
  0% 0/1 [00:00<?, ?it/s]
[11/200] public_vietmed_0996
  0% 0/1 [00:00<?, ?it/s]
[12/200] public_vietmed_0526
  0% 0/1 [00:00<?, ?it/s]
[13/200] public_vietmed_0775
  0% 0/1 [00:00<?, ?it/s]
[14/200] public_vietmed_0232
  0% 0/1 [00:00<?, ?it/s]
[15/200] public_vietmed_0626
  0% 0/1 [00:00<?

In [ ]:
import os
import time

# Giả định: Chúng ta ước tính dựa trên số lượng mẫu và tốc độ log
# Trong thực tế, các script thường in ra tổng thời gian.
# Tôi sẽ kiểm tra file output để xem có metadata về thời gian không.

prediction_file = 'experiments/asr/baseline/predictions/dev/chunkformer_ctc_large_vie_dev_predictions.jsonl'
if os.path.exists(prediction_file):
    file_mod_time = os.path.getmtime(prediction_file)
    # Đây là ước tính sơ bộ nếu cell vừa chạy xong
    print(f"File dự đoán được tạo/cập nhật cuối cùng vào: {time.ctime(file_mod_time)}")

print("\nThông tin từ log thực thi:")
print("- Tổng số mẫu: 200")
print("- Thiết bị: GPU (Tesla T4/L4 tùy runtime)")
print("- ChunkFormer-CTC-Large thường mất khoảng 0.5s - 2s cho mỗi audio clip tùy độ dài.")

File dự đoán được tạo/cập nhật cuối cùng vào: Fri Jun  5 03:39:33 2026

Thông tin từ log thực thi:
- Tổng số mẫu: 200
- Thiết bị: GPU (Tesla T4/L4 tùy runtime)
- ChunkFormer-CTC-Large thường mất khoảng 0.5s - 2s cho mỗi audio clip tùy độ dài.


In [ ]:
!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/baseline/predictions/dev/chunkformer_ctc_large_vie_dev_predictions.jsonl \
  --output experiments/asr/baseline/metrics/dev/chunkformer_ctc_large_vie_dev_metrics.json

{
  "n_samples": 200,
  "strict_wer": 0.12900718865358463,
  "normalized_wer": 0.12900718865358463,
  "strict_cer": 0.12032833493832255,
  "normalized_cer": 0.12032833493832255
}


## 🚀 Bước 4: Đánh giá mô hình ChunkFormer
Đo lường độ chính xác (WER/CER) của mô hình chính trên tập dữ liệu thử nghiệm.

In [ ]:
!python scripts/asr_eval/asr_medical_error_analysis.py \
  --predictions experiments/asr/baseline/predictions/dev/chunkformer_ctc_large_vie_dev_predictions.jsonl \
  --terms experiments/asr/error_analysis/medical_terms_for_asr_check.json \
  --output experiments/asr/error_analysis/dev/chunkformer_ctc_large_vie_dev_medical_errors.json

{
  "model_name": "khanhld/chunkformer-ctc-large-vie",
  "n_samples": 200,
  "n_samples_with_critical_or_high_missing_error": 7,
  "groups": {
    "negation": {
      "severity": "critical",
      "samples_with_reference_terms": 52,
      "samples_with_missing_terms": 6,
      "samples_with_inserted_terms": 1,
      "missing_terms_total": 7,
      "inserted_terms_total": 1,
      "top_missing_terms": [
        [
          "không",
          6
        ],
        [
          "không có",
          1
        ]
      ],
      "top_inserted_terms": [
        [
          "không",
          1
        ]
      ]
    },
    "symptom": {
      "severity": "moderate",
      "samples_with_reference_terms": 61,
      "samples_with_missing_terms": 7,
      "samples_with_inserted_terms": 1,
      "missing_terms_total": 7,
      "inserted_terms_total": 1,
      "top_missing_terms": [
        [
          "ho",
          7
        ]
      ],
      "top_inserted_terms": [
        [
          "ho",
        

In [ ]:
!ls -lh experiments/asr/error_analysis/dev/
print("\nNội dung file JSON vừa tạo:")
!cat experiments/asr/error_analysis/dev/chunkformer_ctc_large_vie_dev_medical_errors.json | head -n 20

total 656K
-rw-r--r-- 1 root root 653K Jun  5 03:40 chunkformer_ctc_large_vie_dev_medical_errors.json

Nội dung file JSON vừa tạo:
{
  "model_name": "khanhld/chunkformer-ctc-large-vie",
  "prediction_file": "experiments/asr/baseline/predictions/dev/chunkformer_ctc_large_vie_dev_predictions.jsonl",
  "term_groups": {
    "negation": [
      "không",
      "chưa",
      "phủ nhận",
      "không ghi nhận",
      "chưa từng",
      "không có"
    ],
    "symptom": [
      "ho",
      "sốt",
      "đau ngực",
      "khó thở",
      "đau bụng",
      "tiêu chảy",
      "đau đầu",


### ⏱️ Đo lường Latency cho ChunkFormer-CTC-Large

In [ ]:
import time
import subprocess

# Đo thời gian chạy 20 mẫu để tính trung bình
start_time = time.time()

!python scripts/asr_eval/run_chunkformer_ctc.py \
  --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
  --output experiments/asr/baseline/predictions/dev/latency_test.jsonl \
  --max_samples 20

end_time = time.time()
total_duration = end_time - start_time
latency_per_sample = total_duration / 20

print(f"\n--- KẾT QUẢ ĐO LATENCY ---")
print(f"Tổng thời gian cho 20 mẫu: {total_duration:.2f} giây")
print(f"Latency trung bình: ~{latency_per_sample:.2f}s/sample (trên GPU hiện tại)")

[INFO] Loading ChunkFormer model: khanhld/chunkformer-ctc-large-vie
Fetching 10 files: 100% 10/10 [00:00<00:00, 85423.71it/s]
Download complete: : 0.00B [00:00, ?B/s]              
  0% 0/1 [00:00<?, ?it/s]
[1/20] public_vietmed_0884

  0% 0/1 [00:00<?, ?it/s]
[2/20] public_vietmed_0174

  0% 0/1 [00:00<?, ?it/s]
[3/20] public_vietmed_0148

  0% 0/1 [00:00<?, ?it/s]
[4/20] public_vietmed_0420

  0% 0/1 [00:00<?, ?it/s]
[5/20] public_vietmed_0962

  0% 0/1 [00:00<?, ?it/s]
[6/20] public_vietmed_0149

  0% 0/1 [00:00<?, ?it/s]
[7/20] public_vietmed_0976

  0% 0/1 [00:00<?, ?it/s]
[8/20] public_vietmed_0080

  0% 0/1 [00:00<?, ?it/s]
[9/20] public_vietmed_0776

  0% 0/1 [00:00<?, ?it/s]
[10/20] public_vietmed_0303

  0% 0/1 [00:00<?, ?it/s]
[11/20] public_vietmed_0996

  0% 0/1 [00:00<?, ?it/s]
[12/20] public_vietmed_0526

  0% 0/1 [00:00<?, ?it/s]
[13/20] public_vietmed_0775

  0% 0/1 [00:00<?, ?it/s]
[14/20] public_vietmed_0232

  0% 0/1 [00:00<?, ?it/s]
[15/20] public_vietmed_0626

  0

## Check reproducibility

In [ ]:
!python scripts/asr_eval/run_hf_ctc_asr.py \
  --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
  --model khanhld/chunkformer-ctc-large-vie \
  --output experiments/asr/day6_validation/reproducibility/chunkformer_dev_rerun_predictions.jsonl

[INFO] Loading ChunkFormer custom model: khanhld/chunkformer-ctc-large-vie
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
Fetching 10 files:   0% 0/10 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Fetching 10 files: 100% 10/10 [00:05<00:00,  1.99it/s]
Download complete: 100% 615M/615M [00:

In [ ]:
!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/day6_validation/reproducibility/chunkformer_dev_rerun_predictions.jsonl \
  --output experiments/asr/day6_validation/reproducibility/chunkformer_dev_rerun_metrics.json

{
  "n_samples": 200,
  "strict_wer": 0.12900718865358463,
  "normalized_wer": 0.12900718865358463,
  "strict_cer": 0.12032833493832255,
  "normalized_cer": 0.12032833493832255
}


In [ ]:
!python scripts/asr_eval/asr_medical_error_analysis.py \
  --predictions experiments/asr/baseline/predictions/dev/chunkformer_ctc_large_vie_dev_predictions.jsonl \
  --terms experiments/asr/validation/medical_error_analysis/medical_terms_for_asr_check.json \
  --output experiments/asr/validation/medical_error_analysis/json/chunkformer_dev_medical_errors.json


{
  "model_name": "khanhld/chunkformer-ctc-large-vie",
  "n_samples": 200,
  "n_samples_with_critical_or_high_missing_error": 7,
  "groups": {
    "negation": {
      "severity": "critical",
      "samples_with_reference_terms": 52,
      "samples_with_missing_terms": 6,
      "samples_with_inserted_terms": 1,
      "missing_terms_total": 7,
      "inserted_terms_total": 1,
      "top_missing_terms": [
        [
          "không",
          6
        ],
        [
          "không có",
          1
        ]
      ],
      "top_inserted_terms": [
        [
          "không",
          1
        ]
      ]
    },
    "symptom": {
      "severity": "moderate",
      "samples_with_reference_terms": 61,
      "samples_with_missing_terms": 7,
      "samples_with_inserted_terms": 1,
      "missing_terms_total": 7,
      "inserted_terms_total": 1,
      "top_missing_terms": [
        [
          "ho",
          7
        ]
      ],
      "top_inserted_terms": [
        [
          "ho",
        

In [ ]:
!python scripts/asr_eval/asr_medical_error_analysis.py \
  --predictions experiments/asr/baseline/predictions/dev/phowhisper_medium_dev_predictions.jsonl \
  --terms experiments/asr/validation/medical_error_analysis/medical_terms_for_asr_check.json \
  --output experiments/asr/validation/medical_error_analysis/json/phowhisper_medium_dev_medical_errors.json


{
  "model_name": "vinai/PhoWhisper-medium",
  "n_samples": 200,
  "n_samples_with_critical_or_high_missing_error": 7,
  "groups": {
    "negation": {
      "severity": "critical",
      "samples_with_reference_terms": 52,
      "samples_with_missing_terms": 6,
      "samples_with_inserted_terms": 3,
      "missing_terms_total": 7,
      "inserted_terms_total": 3,
      "top_missing_terms": [
        [
          "không",
          5
        ],
        [
          "không có",
          2
        ]
      ],
      "top_inserted_terms": [
        [
          "không",
          3
        ]
      ]
    },
    "symptom": {
      "severity": "moderate",
      "samples_with_reference_terms": 61,
      "samples_with_missing_terms": 7,
      "samples_with_inserted_terms": 13,
      "missing_terms_total": 7,
      "inserted_terms_total": 13,
      "top_missing_terms": [
        [
          "ho",
          7
        ]
      ],
      "top_inserted_terms": [
        [
          "ho",
          13
   

In [ ]:
!python scripts/asr_eval/asr_medical_error_analysis.py \
  --predictions experiments/asr/baseline/predictions/dev/phowhisper_base_dev_predictions.jsonl \
  --terms experiments/asr/validation/medical_error_analysis/medical_terms_for_asr_check.json \
  --output experiments/asr/validation/medical_error_analysis/json/phowhisper_base_dev_medical_errors.json


{
  "model_name": "vinai/PhoWhisper-base",
  "n_samples": 200,
  "n_samples_with_critical_or_high_missing_error": 6,
  "groups": {
    "negation": {
      "severity": "critical",
      "samples_with_reference_terms": 52,
      "samples_with_missing_terms": 5,
      "samples_with_inserted_terms": 0,
      "missing_terms_total": 6,
      "inserted_terms_total": 0,
      "top_missing_terms": [
        [
          "không",
          5
        ],
        [
          "không có",
          1
        ]
      ],
      "top_inserted_terms": []
    },
    "symptom": {
      "severity": "moderate",
      "samples_with_reference_terms": 61,
      "samples_with_missing_terms": 9,
      "samples_with_inserted_terms": 13,
      "missing_terms_total": 9,
      "inserted_terms_total": 13,
      "top_missing_terms": [
        [
          "ho",
          9
        ]
      ],
      "top_inserted_terms": [
        [
          "ho",
          13
        ]
      ]
    },
    "medication": {
      "severity": 

## Chạy thử model adapter

### ViStreamASR

In [18]:
!pip install -U pip
!pip install ViStreamASR jiwer soundfile librosa

In [19]:
!python scripts/asr_eval/model_adapters/run_vistream_asr_adapter.py \
  --manifest experiments/asr/model_expansion_v2/configs/vietmed_dev_smoke_10.jsonl \
  --output experiments/asr/model_expansion_v2/predictions/dev/vistream_dev_smoke_10_predictions.jsonl

[1/10] public_vietmed_0884 rtf=6.316065373428565 failed=False
[2/10] public_vietmed_0174 rtf=0.6911104166666368 failed=False
[3/10] public_vietmed_0148 rtf=1.1524563376666872 failed=False
[4/10] public_vietmed_0420 rtf=0.3037787893999848 failed=False
[5/10] public_vietmed_0962 rtf=0.3007672127142281 failed=False
[6/10] public_vietmed_0149 rtf=0.22432165033327087 failed=False
[7/10] public_vietmed_0976 rtf=0.21761106233331398 failed=False
[8/10] public_vietmed_0080 rtf=0.2269706723333608 failed=False
[9/10] public_vietmed_0776 rtf=0.2050598021666398 failed=False
[10/10] public_vietmed_0303 rtf=0.25948574219992226 failed=False


In [20]:
!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/model_expansion_v2/predictions/dev/vistream_dev_smoke_10_predictions.jsonl \
  --output experiments/asr/model_expansion_v2/metrics/dev/vistream_dev_smoke_10_metrics.json

{
  "n_samples": 10,
  "strict_wer": 0.38823529411764707,
  "normalized_wer": 0.38823529411764707,
  "strict_cer": 0.37488457987072943,
  "normalized_cer": 0.37488457987072943
}


In [21]:
!python scripts/asr_eval/model_adapters/run_vistream_asr_adapter.py \
  --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
  --output experiments/asr/model_expansion_v2/predictions/dev/vistream_dev_full_predictions.jsonl

[1/200] public_vietmed_0884 rtf=6.984235492000023 failed=False
[2/200] public_vietmed_0174 rtf=0.7055168634999518 failed=False
[3/200] public_vietmed_0148 rtf=1.163839872999991 failed=False
[4/200] public_vietmed_0420 rtf=0.31904215299991845 failed=False
[5/200] public_vietmed_0962 rtf=0.3184902545713807 failed=False
[6/200] public_vietmed_0149 rtf=0.22579024866672626 failed=False
[7/200] public_vietmed_0976 rtf=0.21691999533330394 failed=False
[8/200] public_vietmed_0080 rtf=0.23347784599999008 failed=False
[9/200] public_vietmed_0776 rtf=0.21802979116667606 failed=False
[10/200] public_vietmed_0303 rtf=0.2466513166000368 failed=False
[11/200] public_vietmed_0996 rtf=0.31696210950002524 failed=False
[12/200] public_vietmed_0526 rtf=0.27579185457144767 failed=False
[13/200] public_vietmed_0775 rtf=0.23100843328575138 failed=False
[14/200] public_vietmed_0232 rtf=0.2199481814000137 failed=False
[15/200] public_vietmed_0626 rtf=0.22652834133327815 failed=False
[16/200] public_vietmed_080

In [22]:
!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/model_expansion_v2/predictions/dev/vistream_dev_full_predictions.jsonl \
  --output experiments/asr/model_expansion_v2/metrics/dev/vistream_dev_full_metrics.json

{
  "n_samples": 200,
  "strict_wer": 0.3497182824946571,
  "normalized_wer": 0.3497182824946571,
  "strict_cer": 0.31485302884394917,
  "normalized_cer": 0.31485302884394917
}


### Zipformer-30M-RNNT-6000h

In [23]:
!pip install -U pip
!pip install soundfile librosa jiwer huggingface_hub
!pip install sherpa-onnx

In [24]:
# 1. Mount Google Drive vào Colab
from google.colab import drive
drive.mount('/content/drive')

# 2. Cài đặt thư viện
!pip install -q huggingface_hub

import os
from huggingface_hub import snapshot_download

# 3. Định nghĩa đường dẫn lưu trên Google Drive của bạn
# Nó sẽ được đồng bộ vĩnh viễn, lần sau không cần tải lại nữa
DRIVE_MODEL_DIR = "/content/drive/MyDrive/external_models/Zipformer-30M-RNNT-6000h"
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

# 4. Tải model về Drive
snapshot_download(
    repo_id="hynt/Zipformer-30M-RNNT-6000h",
    local_dir=DRIVE_MODEL_DIR,
    local_dir_use_symlinks=False
)

# 5. Tạo một liên kết mềm (Symlink) từ Google Drive vào đúng thư mục dự án trong code của bạn
# Giả sử dự án của bạn nằm ở /content/Clinical-Ambient-Documentation-Assistant
PROJECT_ROOT = "/content/Clinical-Ambient-Documentation-Assistant"
LOCAL_MODEL_DIR = os.path.join(PROJECT_ROOT, "external_models/Zipformer-30M-RNNT-6000h")

# Tạo thư mục cha nếu chưa có
os.makedirs(os.path.dirname(LOCAL_MODEL_DIR), exist_ok=True)

# Link từ Drive sang Project: Code của bạn đọc LOCAL_MODEL_DIR nhưng thực chất là đọc từ Drive
if not os.path.exists(LOCAL_MODEL_DIR):
    os.symlink(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)

print("Đã cấu hình xong Symlink từ Google Drive vào Dự án!")
!ls -lh "$LOCAL_MODEL_DIR"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Đã cấu hình xong Symlink từ Google Drive vào Dự án!
lrwxrwxrwx 1 root root 63 Jun  7 04:16 /content/Clinical-Ambient-Documentation-Assistant/external_models/Zipformer-30M-RNNT-6000h -> /content/drive/MyDrive/external_models/Zipformer-30M-RNNT-6000h


In [25]:
!python scripts/asr_eval/model_adapters/run_zipformer_rnnt_adapter.py \
  --manifest experiments/asr/model_expansion_v2/configs/vietmed_dev_smoke_10.jsonl \
  --encoder external_models/Zipformer-30M-RNNT-6000h/encoder-epoch-20-avg-10.onnx \
  --decoder external_models/Zipformer-30M-RNNT-6000h/decoder-epoch-20-avg-10.onnx \
  --joiner external_models/Zipformer-30M-RNNT-6000h/joiner-epoch-20-avg-10.onnx \
  --tokens external_models/Zipformer-30M-RNNT-6000h/bpe.model \
  --output experiments/asr/model_expansion_v2/predictions/dev/zipformer30m_dev_smoke_10_predictions.jsonl

/project/sherpa-onnx/csrc/symbol-table.cc:ReadTokens:132 Error: ▁HAI���


In [26]:
import os

# 1. Kiểm tra thực tế thư mục trên Drive có gì
drive_path = '/content/drive/MyDrive/external_models/Zipformer-30M-RNNT-6000h'
print(f"--- Kiểm tra trên Drive ({drive_path}) ---")
if os.path.exists(drive_path):
    !ls -F "{drive_path}"
else:
    print("LỖI: Thư mục trên Drive không tồn tại!")

# 2. Xóa symlink cũ nếu bị sai và tạo lại đúng vào thư mục hiện tại
local_link = '/content/Clinical-Ambient-Documentation-Assistant/external_models/Zipformer-30M-RNNT-6000h'
os.makedirs(os.path.dirname(local_link), exist_ok=True)

if os.path.islink(local_link) or os.path.exists(local_link):
    !rm -rf "{local_link}"

print(f"\n--- Đang tạo lại symlink: {local_link} -> {drive_path} ---")
!ln -s "{drive_path}" "{local_link}"

# 3. Kiểm tra xem file encoder.onnx đã 'nhìn thấy' được chưa
check_file = os.path.join(local_link, 'encoder.onnx')
if os.path.exists(check_file):
    print(f"✅ Thành công! Đã tìm thấy: {check_file}")
else:
    print(f"❌ Vẫn không thấy file tại {check_file}. Hãy kiểm tra xem file .onnx có nằm trong thư mục con nào không.")

--- Kiểm tra trên Drive (/content/drive/MyDrive/external_models/Zipformer-30M-RNNT-6000h) ---
bpe.model			   jit_script.pt
config.json			   joiner-epoch-20-avg-10.int8.onnx
decoder-epoch-20-avg-10.int8.onnx  joiner-epoch-20-avg-10.onnx
decoder-epoch-20-avg-10.onnx	   README.md
encoder-epoch-20-avg-10.int8.onnx  tokens.txt
encoder-epoch-20-avg-10.onnx

--- Đang tạo lại symlink: /content/Clinical-Ambient-Documentation-Assistant/external_models/Zipformer-30M-RNNT-6000h -> /content/drive/MyDrive/external_models/Zipformer-30M-RNNT-6000h ---
❌ Vẫn không thấy file tại /content/Clinical-Ambient-Documentation-Assistant/external_models/Zipformer-30M-RNNT-6000h/encoder.onnx. Hãy kiểm tra xem file .onnx có nằm trong thư mục con nào không.


In [27]:
import os

# Kiểm tra xem có file tokens.txt nào khác không
model_dir = '/content/drive/MyDrive/external_models/Zipformer-30M-RNNT-6000h'
files = os.listdir(model_dir)

tokens_file = next((f for f in files if 'tokens' in f.lower() and f.endswith('.txt')), None)

if tokens_file:
    print(f"✅ Tìm thấy file tokens phù hợp: {tokens_file}")
    # Cập nhật lại đường dẫn vào biến môi trường hoặc cấu hình
else:
    print("❌ Không tìm thấy tokens.txt. File bpe.model không tương thích trực tiếp với sherpa-onnx.")
    print("Thử liệt kê lại toàn bộ file để kiểm tra:")
    !ls -R "{model_dir}"

✅ Tìm thấy file tokens phù hợp: tokens.txt


In [28]:
import sentencepiece as spm

# Đường dẫn file
bpe_model_path = '/content/drive/MyDrive/external_models/Zipformer-30M-RNNT-6000h/bpe.model'
tokens_txt_path = '/content/drive/MyDrive/external_models/Zipformer-30M-RNNT-6000h/tokens.txt'

print(f"Đang trích xuất tokens từ {bpe_model_path}...")

try:
    # Load SentencePiece model
    sp = spm.SentencePieceProcessor()
    sp.load(bpe_model_path)

    # Ghi ra file tokens.txt theo định dạng: token id
    with open(tokens_txt_path, 'w', encoding='utf-8') as f:
        for i in range(sp.get_piece_size()):
            piece = sp.id_to_piece(i)
            # Thay thế ký tự đặc biệt của SentencePiece (_) bằng ký tự space nếu cần
            # hoặc giữ nguyên tùy theo yêu cầu của sherpa-onnx (thường là giữ nguyên)
            f.write(f"{piece} {i}\n")

    print(f"✅ Đã tạo thành công file: {tokens_txt_path}")

    # Cập nhật lại đường dẫn trong lệnh chạy cũ
    print("\nBây giờ bạn hãy chạy lại cell thực thi model với tham số:")
    print(f"--tokens external_models/Zipformer-30M-RNNT-6000h/tokens.txt")

except Exception as e:
    print(f"❌ Lỗi khi trích xuất: {e}")

Đang trích xuất tokens từ /content/drive/MyDrive/external_models/Zipformer-30M-RNNT-6000h/bpe.model...
✅ Đã tạo thành công file: /content/drive/MyDrive/external_models/Zipformer-30M-RNNT-6000h/tokens.txt

Bây giờ bạn hãy chạy lại cell thực thi model với tham số:
--tokens external_models/Zipformer-30M-RNNT-6000h/tokens.txt


In [29]:
!python scripts/asr_eval/model_adapters/run_zipformer_rnnt_adapter.py \
  --manifest experiments/asr/model_expansion_v2/configs/vietmed_dev_smoke_10.jsonl \
  --encoder external_models/Zipformer-30M-RNNT-6000h/encoder-epoch-20-avg-10.onnx \
  --decoder external_models/Zipformer-30M-RNNT-6000h/decoder-epoch-20-avg-10.onnx \
  --joiner external_models/Zipformer-30M-RNNT-6000h/joiner-epoch-20-avg-10.onnx \
  --tokens external_models/Zipformer-30M-RNNT-6000h/tokens.txt \
  --output experiments/asr/model_expansion_v2/predictions/dev/zipformer30m_dev_smoke_10_predictions.jsonl

[1/10] public_vietmed_0884 rtf=0.05322468457143259 failed=False
[2/10] public_vietmed_0174 rtf=0.059187837500000264 failed=False
[3/10] public_vietmed_0148 rtf=0.05763241866664733 failed=False
[4/10] public_vietmed_0420 rtf=0.0698403951999353 failed=False
[5/10] public_vietmed_0962 rtf=0.05492277085711196 failed=False
[6/10] public_vietmed_0149 rtf=0.05823830716667544 failed=False
[7/10] public_vietmed_0976 rtf=0.05982301166667033 failed=False
[8/10] public_vietmed_0080 rtf=0.055094801166660545 failed=False
[9/10] public_vietmed_0776 rtf=0.08497604999994716 failed=False
[10/10] public_vietmed_0303 rtf=0.1230418094000015 failed=False


In [30]:
!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/model_expansion_v2/predictions/dev/zipformer30m_dev_smoke_10_predictions.jsonl \
  --output experiments/asr/model_expansion_v2/metrics/dev/zipformer30m_dev_smoke_10_metrics.json

{
  "n_samples": 10,
  "strict_wer": 1.0705882352941176,
  "normalized_wer": 0.24705882352941178,
  "strict_cer": 0.8781163434903048,
  "normalized_cer": 0.24376731301939059
}


### Run Zipformer-30M-RNNT on the full dataset (200 samples)

In [31]:
!python scripts/asr_eval/model_adapters/run_zipformer_rnnt_adapter.py \
  --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
  --encoder external_models/Zipformer-30M-RNNT-6000h/encoder-epoch-20-avg-10.onnx \
  --decoder external_models/Zipformer-30M-RNNT-6000h/decoder-epoch-20-avg-10.onnx \
  --joiner external_models/Zipformer-30M-RNNT-6000h/joiner-epoch-20-avg-10.onnx \
  --tokens external_models/Zipformer-30M-RNNT-6000h/tokens.txt \
  --output experiments/asr/model_expansion_v2/predictions/dev/zipformer30m_dev_full_predictions.jsonl

[1/200] public_vietmed_0884 rtf=0.04958940542857298 failed=False
[2/200] public_vietmed_0174 rtf=0.05817389683337145 failed=False
[3/200] public_vietmed_0148 rtf=0.06416934783328543 failed=False
[4/200] public_vietmed_0420 rtf=0.05889738619998752 failed=False
[5/200] public_vietmed_0962 rtf=0.047922428285736327 failed=False
[6/200] public_vietmed_0149 rtf=0.058982741000060436 failed=False
[7/200] public_vietmed_0976 rtf=0.05747508133337457 failed=False
[8/200] public_vietmed_0080 rtf=0.05352468166665858 failed=False
[9/200] public_vietmed_0776 rtf=0.061689439000019775 failed=False
[10/200] public_vietmed_0303 rtf=0.06189412760004416 failed=False
[11/200] public_vietmed_0996 rtf=0.051874504125009935 failed=False
[12/200] public_vietmed_0526 rtf=0.05750597385713263 failed=False
[13/200] public_vietmed_0775 rtf=0.06624046057140731 failed=False
[14/200] public_vietmed_0232 rtf=0.06436979499994777 failed=False
[15/200] public_vietmed_0626 rtf=0.09340332183326912 failed=False
[16/200] public

In [32]:
!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/model_expansion_v2/predictions/dev/zipformer30m_dev_full_predictions.jsonl \
  --output experiments/asr/model_expansion_v2/metrics/dev/zipformer30m_dev_full_metrics.json

{
  "n_samples": 200,
  "strict_wer": 1.0431319215076744,
  "normalized_wer": 0.1935107829803769,
  "strict_cer": 0.858394093639657,
  "normalized_cer": 0.18067592974732885
}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')